# Check a boundary before trusting a chain law

AuthorityEffect: None.

> **Claim-typing legend**
> - **[Pinned]**: reproduced from verified checked-in Grammar 000 fixtures.
> - **[Executed]**: verified by deterministic Python execution in this notebook.
> - **[Declared]**: pedagogical convention or beginner framing.
> - **[Open]**: questions left for future exploration.


## Start here

A filled triangle needs all three edges. With that boundary present, differences of vertex potentials cancel around the face. The carrier must reject a face with an absent boundary edge before the chain-law audit.

## Exact model

d1(d0(f))(a,b,c) = (f(c)-f(b)) - (f(c)-f(a)) + (f(b)-f(a)) = 0.

All numerical fixtures use integers or Fraction. A passing run means the named finite checks passed.


In [ ]:
import importlib.util
import sys
from fractions import Fraction
from pathlib import Path

ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "education/scripts/build_system_integration.py").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from education.scripts.system_integration_support import (
    expect_error,
)


def load_source(name, relative):
    spec = importlib.util.spec_from_file_location(name, ROOT / relative)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

## Worked computation

Run the cell, then compare the output with the exact model above.


In [ ]:
from pfp.domain.ddc import (
    Cochain0,
    Edge,
    Face,
    SimplicialComplex,
    audit_nilpotency_all_basis,
    d0,
    d1,
)

vertices = ("a", "b", "c")
edges = (Edge("a", "b"), Edge("b", "c"), Edge("a", "c"))
carrier = SimplicialComplex(vertices, edges, (Face(*vertices),))
report = audit_nilpotency_all_basis(carrier)
assert report.is_nilpotent and report.num_faces_evaluated == 3
for values in [
    (Fraction(0), Fraction(2, 3), Fraction(-5, 7)),
    (Fraction(1), Fraction(0), Fraction(0)),
]:
    f = Cochain0(dict(zip(vertices, values)))
    assert d1(carrier, d0(carrier, f)).get(*vertices) == 0
print("Three basis potentials and two additional exact fixtures satisfy the chain law.")

## Exercise and adversarial check

Remove edge (a,c) while keeping the face. Expect constructor refusal. Then reverse every edge and check the full basis again.

Predict the result before running the reference check below. Each assertion checks a specific value or failure mode.


In [ ]:
missing = expect_error(
    ValueError, lambda: SimplicialComplex(vertices, edges[:2], (Face(*vertices),))
)
assert "boundary edge" in missing and "absent in both orientations" in missing
reversed_edges = tuple(Edge(edge.v, edge.u) for edge in edges)
reversed_carrier = SimplicialComplex(vertices, reversed_edges, (Face(*vertices),))
assert audit_nilpotency_all_basis(reversed_carrier).is_nilpotent
expect_error(ValueError, lambda: Face("a", "a", "c"))
print(
    "The constructor rejects a missing boundary and the basis audit supports reversed edge orientations."
)

## Evidence boundary

The constructor checks boundary completeness, and the independent basis audit checks the chain law. Neither check replaces the other.

## Source route

- [symbol_game/domain/ddc/complex.py](../../../../../symbol_game/domain/ddc/complex.py)
- [symbol_game/domain/ddc/cochains.py](../../../../../symbol_game/domain/ddc/cochains.py)
- [symbol_game/domain/ddc/coboundary.py](../../../../../symbol_game/domain/ddc/coboundary.py)

Return to the [intermediate integration route](../verified_route.md) and [curriculum overview](../curriculum_25_200.md).
